In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install opencv-python-headless numpy --quiet

In [3]:
!pip install ultralytics --quiet

In [4]:
%%writefile config.py
DETECT_ADDR  = ("127.0.0.1", 7001)
WRITER_ADDR  = ("127.0.0.1", 7002)
DEFAULT_SOURCE   = "0"
MAX_FRAMES   = 99999
SEND_FPS     = 25.0

JPG_QUALITY  = 80
OUTPUT_DIR    = "/content/drive/MyDrive/LAB5/output"

Overwriting config.py


In [5]:
%%writefile /content/drive/MyDrive/LAB5/net_utils.py
import base64, json, socket, time
from typing import Any, Dict, Generator

def to_b64(data: bytes) -> str:
    return base64.b64encode(data).decode()

def from_b64(s: str) -> bytes:
    return base64.b64decode(s)

def write_msg(sock: socket.socket, obj: Dict[str, Any]) -> None:
    raw = (json.dumps(obj) + "\n").encode()
    sock.sendall(raw)

def read_msgs(sock: socket.socket) -> Generator[Dict, None, None]:
    reader = sock.makefile("r", encoding="utf-8")
    for line in reader:
        line = line.strip()
        if line:
            yield json.loads(line)

class TCPServer:
    def __init__(self, addr: tuple):
        self.addr = addr
        self._sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self._sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        self._sock.bind(addr)
        self._sock.listen(1)
        print(f"[TCPServer] Đang lắng nghe {addr}")

    def wait_for_client(self):
        conn, who = self._sock.accept()
        print(f"[TCPServer] Có kết nối từ {who}")
        return conn

    def close(self):
        self._sock.close()

class TCPClient:
    def __init__(self, addr: tuple, retry: int = 12, gap: float = 1.0):
        self.addr = addr
        self.conn = self._connect(retry, gap)

    def _connect(self, retry, gap) -> socket.socket:
        for i in range(1, retry + 1):
            try:
                s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
                s.connect(self.addr)
                print(f"[TCPClient] Kết nối {self.addr}")
                return s
            except ConnectionRefusedError:
                print(f"[TCPClient] Thử {i}/{retry} …")
                time.sleep(gap)
        raise RuntimeError(f"Hết retry, không kết nối được {self.addr}")

    def send(self, obj: Dict[str, Any]) -> None:
        write_msg(self.conn, obj)

    def recv_all(self) -> Generator[Dict, None, None]:
        yield from read_msgs(self.conn)

    def close(self):
        self.conn.close()

Overwriting /content/drive/MyDrive/LAB5/net_utils.py


In [6]:
%%writefile /content/drive/MyDrive/LAB5/node_capture.py
import time
from datetime import datetime, timezone
import cv2, numpy as np
import config
from net_utils import TCPClient, to_b64


class CaptureNode:
    def __init__(self, source: str, max_frames: int, fps: float):
        self.source     = source
        self.max_frames = max_frames
        self.interval   = 1.0 / fps if fps > 0 else 0

    def _fake(self):
        for i in range(self.max_frames):
            img = np.full((480, 640, 3), 30, dtype=np.uint8)
            ox  = 50 + (i * 15) % 480
            cv2.ellipse(img, (ox+40, 80), (40, 50), 0, 0, 360, (210,210,210), -1)
            cv2.rectangle(img, (ox, 130), (ox+80, 360), (150,150,150), -1)
            cv2.putText(img, f"SIM {i}", (ox, 400),
                        cv2.FONT_HERSHEY_PLAIN, 1.4, (0,255,100), 2)
            yield i, img

    def _real(self):
        src = int(self.source) if self.source.isdigit() else self.source
        cap = cv2.VideoCapture(src)
        if not cap.isOpened():
            print("Không mở được nguồn → dùng frame giả")
            yield from self._fake(); return
        idx = 0
        try:
            while idx < self.max_frames:
                ret, frame = cap.read()
                if not ret: break
                yield idx, frame
                idx += 1
        finally:
            cap.release()

    def _encode(self, frame: np.ndarray) -> str:
        flags = [cv2.IMWRITE_JPEG_QUALITY, config.JPG_QUALITY]
        _, buf = cv2.imencode(".jpg", frame, flags)
        return to_b64(buf.tobytes())

    def stream_to(self, client: TCPClient) -> None:
        for idx, frame in self._real():
            pkt = {
                "kind":   "frame",
                "cam":    "A1",
                "seq":    idx,
                "ts":     datetime.now(timezone.utc).isoformat(),
                "enc":    "jpg",
                "data":   self._encode(frame),
                "W":      frame.shape[1],
                "H":      frame.shape[0],
            }
            client.send(pkt)
            print(f"[Capture] seq={idx} gửi xong")
            if self.interval:
                time.sleep(self.interval)
        client.send({"kind": "done"})
        print("[Capture] Hoàn thành")


def run(source=config.DEFAULT_SOURCE,
        max_frames=config.MAX_FRAMES,
        fps=config.SEND_FPS):
    cli  = TCPClient(config.DETECT_ADDR)
    node = CaptureNode(source, max_frames, fps)
    try:
        node.stream_to(cli)
    finally:
        cli.close()

Overwriting /content/drive/MyDrive/LAB5/node_capture.py


In [7]:
%%writefile /content/drive/MyDrive/LAB5/node_detect.py
from datetime import datetime, timezone
from typing import Dict, List
import cv2, numpy as np
import config
from net_utils import TCPServer, TCPClient, from_b64, write_msg, read_msgs
from ultralytics import YOLO


class PersonDetector:
    def __init__(self):
        self._model = YOLO("yolov8n.pt")

    def run(self, frame: np.ndarray) -> List[Dict]:
        # class 0 = person trong COCO
        results = self._model(frame, classes=[0], verbose=False, conf=0.5, iou=0.4)[0]
        out = []
        for box in results.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            conf = float(box.conf[0])
            out.append({
                "x": int(x1), "y": int(y1),
                "w": int(x2 - x1), "h": int(y2 - y1),
                "score": round(conf, 3),
            })
        return out


def _decode_frame(pkt: Dict) -> np.ndarray:
    raw = from_b64(pkt["data"])
    arr = np.frombuffer(raw, np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)


def run():
    writer_cli = TCPClient(config.WRITER_ADDR)
    cam_srv    = TCPServer(config.DETECT_ADDR)
    cam_conn   = cam_srv.wait_for_client()
    detector   = PersonDetector()
    print("[Detect] ✓ YOLO model loaded")

    try:
        for pkt in read_msgs(cam_conn):
            if pkt.get("kind") == "done":
                write_msg(writer_cli.conn, {"kind": "done"})
                print("[Detect] ✓ done → writer")
                break

            frame  = _decode_frame(pkt)
            boxes  = detector.run(frame)
            result = {
                "kind":     "result",
                "cam":      pkt.get("cam"),
                "seq":      pkt.get("seq"),
                "src_ts":   pkt.get("ts"),
                "det_ts":   datetime.now(timezone.utc).isoformat(),
                "n_people": len(boxes),
                "boxes":    boxes,
                "W":        pkt.get("W"),
                "H":        pkt.get("H"),
                "data":     pkt.get("data"),
            }
            write_msg(writer_cli.conn, result)
            print(f"[Detect] seq={result['seq']} → {result['n_people']} người")
    finally:
        cam_conn.close()
        writer_cli.close()
        cam_srv.close()

Overwriting /content/drive/MyDrive/LAB5/node_detect.py


In [8]:
%%writefile /content/drive/MyDrive/LAB5/node_writer.py
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List
import cv2, numpy as np
import config
from net_utils import TCPServer, read_msgs, from_b64


class VideoWriter:
    def __init__(self, path: Path, W: int, H: int, fps: float = 5.0):
        path.parent.mkdir(parents=True, exist_ok=True)
        # lưu AVI trước, sau đó convert sang MP4
        self.avi_path = path.with_suffix(".avi")
        self.mp4_path = path
        fourcc   = cv2.VideoWriter_fourcc(*"XVID")
        self._vw = cv2.VideoWriter(str(self.avi_path), fourcc, fps, (W, H))
        self.fps = fps

    def write(self, frame: np.ndarray) -> None:
        self._vw.write(frame)

    def close(self) -> None:
        self._vw.release()
        # convert AVI → MP4 bằng ffmpeg (có sẵn trên Colab)
        import subprocess
        cmd = [
            "ffmpeg", "-y",
            "-i", str(self.avi_path),
            "-vcodec", "libx264",
            "-crf", "23",
            str(self.mp4_path)
        ]
        result = subprocess.run(cmd, capture_output=True)
        if result.returncode == 0:
            self.avi_path.unlink()  # xóa file AVI tạm
            print(f"[Writer] 🎬 Video: {self.mp4_path}")
        else:
            print(f"[Writer] ⚠ ffmpeg lỗi, dùng file AVI: {self.avi_path}")
            print(result.stderr.decode())


def draw_boxes(frame: np.ndarray, boxes: List[Dict]) -> np.ndarray:
    out = frame.copy()
    for b in boxes:
        x, y, w, h = b["x"], b["y"], b["w"], b["h"]
        cv2.rectangle(out, (x, y), (x+w, y+h), (0, 255, 80), 2)
        cv2.putText(out, f"{b.get('score',0):.2f}", (x, max(y-8,12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,80), 2)
    cv2.putText(out, f"People: {len(boxes)}", (12, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,200,255), 2)
    return out


def decode_frame(b64: str) -> np.ndarray:
    arr = np.frombuffer(from_b64(b64), np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)


class WriterNode:
    def __init__(self, root: str):
        self.root = Path(root)
        self._vw  = None

    def _jsonl_path(self, ts: str) -> Path:
        try:
            day = datetime.fromisoformat(ts).date().isoformat()
        except Exception:
            day = datetime.now(timezone.utc).date().isoformat()
        return self.root / f"date={day}" / "results.jsonl"

    def _init_video(self, record: Dict) -> None:
        day   = datetime.now(timezone.utc).date().isoformat()
        vpath = self.root / f"date={day}" / "annotated.mp4"
        self._vw = VideoWriter(vpath, record.get("W",640),
                               record.get("H",480), config.SEND_FPS)

    def handle(self, record: Dict) -> None:
        if self._vw is None:
            self._init_video(record)

        if record.get("data"):
            frame = decode_frame(record["data"])
            self._vw.write(draw_boxes(frame, record.get("boxes", [])))

        row = {
            "camera_id":      record.get("cam"),
            "frame_id":       record.get("seq"),
            "people_count":   record.get("n_people"),
            "bounding_boxes": record.get("boxes", []),
            "source_ts":      record.get("src_ts"),
            "processed_ts":   record.get("det_ts"),
            "image_width":    record.get("W"),
            "image_height":   record.get("H"),
        }
        ts    = str(row["processed_ts"] or datetime.now(timezone.utc).isoformat())
        jpath = self._jsonl_path(ts)
        jpath.parent.mkdir(parents=True, exist_ok=True)
        with jpath.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(row, ensure_ascii=False) + "\n")

    def finish(self):
        if self._vw:
            self._vw.close()


def run(root: str = config.OUTPUT_DIR):
    srv   = TCPServer(config.WRITER_ADDR)
    conn  = srv.wait_for_client()
    node  = WriterNode(root)
    saved = 0
    try:
        for record in read_msgs(conn):
            if record.get("kind") == "done":
                print("[Writer] ✓ Nhận done.")
                break
            node.handle(record)
            saved += 1
            print(f"[Writer] seq={record.get('seq')} | {record.get('n_people')} người")
    finally:
        node.finish()
        conn.close()
        srv.close()
    print(f"[Writer] Đã lưu {saved} record.")

Overwriting /content/drive/MyDrive/LAB5/node_writer.py


In [9]:
import threading, time, sys

for mod in ["config","node_capture","node_detect","node_writer","net_utils"]:
    sys.modules.pop(mod, None)

import node_writer, node_detect, node_capture

t1 = threading.Thread(target=node_writer.run, daemon=True, name="writer")
t1.start()
time.sleep(1.5)

t2 = threading.Thread(target=node_detect.run, daemon=True, name="detect")
t2.start()
time.sleep(2.0)

t3 = threading.Thread(
    target=node_capture.run,
    kwargs={"source": "/content/drive/MyDrive/LAB5/video.mp4", "max_frames": 99999, "fps": 25.0},
    daemon=True, name="capture"
)
t3.start()

t3.join()
t2.join(timeout=300)
t1.join(timeout=300)
print("\nPipeline xong!")

[TCPServer] Đang lắng nghe ('127.0.0.1', 7002)
[TCPClient] Kết nối ('127.0.0.1', 7002)
[TCPServer] Có kết nối từ ('127.0.0.1', 44140)
[TCPServer] Đang lắng nghe ('127.0.0.1', 7001)
[TCPClient] Kết nối ('127.0.0.1', 7001)[TCPServer] Có kết nối từ ('127.0.0.1', 46658)

[Capture] seq=0 gửi xong
[Capture] seq=1 gửi xong
[Capture] seq=2 gửi xong
[Capture] seq=3 gửi xong
[Capture] seq=4 gửi xong
[Capture] seq=5 gửi xong
[Capture] seq=6 gửi xong
[Capture] seq=7 gửi xong
[Capture] seq=8 gửi xong
[Capture] seq=9 gửi xong
[Capture] seq=10 gửi xong
[Capture] seq=11 gửi xong
[Capture] seq=12 gửi xong
[Capture] seq=13 gửi xong
[Capture] seq=14 gửi xong
[Capture] seq=15 gửi xong
[Capture] seq=16 gửi xong
[Capture] seq=17 gửi xong
[Capture] seq=18 gửi xong
[Detect] seq=0 → 13 người
[Capture] seq=19 gửi xong
[Capture] seq=20 gửi xong
[Capture] seq=21 gửi xong
[Writer] seq=0 | 13 người
[Capture] seq=22 gửi xong
[Capture] seq=23 gửi xong
[Capture] seq=24 gửi xong
[Capture] seq=25 gửi xong
[Capture] seq=

In [10]:
!ps aux | grep -E "storage_node|detector_node" | grep -v grep

In [11]:
import os
print(os.listdir("/content"))

['.config', '__pycache__', 'config.py', 'drive', 'node_detect.py', '=3.5', 'pipeline_out', 'net_utils.py', 'node_capture.py', 'node_writer.py', 'video.mp4', '.ipynb_checkpoints', 'sample_data']


In [12]:
!python camera_node.py --source /content/drive/MyDrive/LAB5/video.mp4 --limit 1394 --fps 10

python3: can't open file '/content/camera_node.py': [Errno 2] No such file or directory


In [13]:
import json, glob
from pathlib import Path

# jsonl
for f in sorted(glob.glob("pipeline_out/**/*.jsonl", recursive=True)):
    print(f"\n{f}")
    with open(f) as fh:
        for line in fh:
            r = json.loads(line)
            print(f"  seq={r.get('seq'):>3} | người={r.get('n_people')} | {r.get('det_ts')}")

# video
for f in sorted(glob.glob("pipeline_out/**/*.mp4", recursive=True)):
    print(f"\nVideo output: {f}")


pipeline_out/2026-06-16/detections.jsonl
  seq=  0 | người=0 | 2026-06-16T22:41:23.950908+00:00
  seq=  1 | người=0 | 2026-06-16T22:41:23.993726+00:00
  seq=  2 | người=0 | 2026-06-16T22:41:24.035738+00:00
  seq=  3 | người=0 | 2026-06-16T22:41:24.077078+00:00
  seq=  4 | người=0 | 2026-06-16T22:41:24.121469+00:00
  seq=  5 | người=0 | 2026-06-16T22:41:24.164055+00:00
  seq=  6 | người=0 | 2026-06-16T22:41:24.207757+00:00
  seq=  7 | người=0 | 2026-06-16T22:41:24.250843+00:00
  seq=  8 | người=0 | 2026-06-16T22:41:24.291939+00:00
  seq=  9 | người=0 | 2026-06-16T22:41:24.332907+00:00
  seq= 10 | người=0 | 2026-06-16T22:41:24.374157+00:00
  seq= 11 | người=0 | 2026-06-16T22:41:24.416825+00:00
  seq= 12 | người=0 | 2026-06-16T22:41:24.457658+00:00
  seq= 13 | người=0 | 2026-06-16T22:41:24.501750+00:00
  seq= 14 | người=0 | 2026-06-16T22:41:24.547298+00:00
  seq= 15 | người=0 | 2026-06-16T22:41:24.585345+00:00
  seq= 16 | người=0 | 2026-06-16T22:41:24.627837+00:00
  seq= 17 | người=0 | 2

In [14]:
!pip install pyspark>=3.5 --quiet

In [15]:
%%writefile /content/drive/MyDrive/LAB5/analytics_spark.py
import argparse
try:
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
except ImportError as e:
    raise SystemExit("Chạy: pip install pyspark>=3.5") from e


def load_data(spark, input_dir: str):
    df = spark.read.json(input_dir)
    # thêm cột giờ từ processed_ts
    df = df.withColumn(
        "hour",
        F.hour(F.to_timestamp("processed_ts"))
    )
    return df


def report_by_camera(df) -> None:
    print("\nThống kê theo camera:")
    df.groupBy("camera_id").agg(
        F.count("frame_id").alias("total_frames"),
        F.sum("people_count").alias("total_detections"),
        F.round(F.avg("people_count"), 2).alias("avg_per_frame"),
        F.max("people_count").alias("peak_count"),
    ).orderBy("camera_id").show(truncate=False)


def report_by_hour(df) -> None:
    print("\nKhung giờ đông người nhất:")
    df.groupBy("camera_id", "hour").agg(
        F.round(F.avg("people_count"), 2).alias("avg_people"),
        F.sum("people_count").alias("total_people"),
    ).orderBy("camera_id", F.desc("avg_people")).show(20, truncate=False)


def report_top_frames(df, n: int = 5) -> None:
    print(f"\nTop {n} frame đông người nhất:")
    df.select("camera_id", "frame_id", "people_count", "processed_ts") \
      .orderBy(F.desc("people_count")) \
      .limit(n) \
      .show(truncate=False)


def summarize(input_dir: str, top_n: int) -> None:
    spark = (
        SparkSession.builder
        .appName("CameraAnalytics")
        .master("local[*]")
        .config("spark.sql.session.timeZone", "UTC")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")  # tắt log rác
    try:
        df = load_data(spark, input_dir)
        print(f"✓ Đọc được {df.count()} bản ghi từ {input_dir}")
        report_by_camera(df)
        report_by_hour(df)
        report_top_frames(df, top_n)
    finally:
        spark.stop()


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="Phân tích dữ liệu đếm người bằng PySpark")
    p.add_argument("--input",  default="pipeline_out", help="Thư mục chứa JSONL")
    p.add_argument("--top",    type=int, default=5,    help="Số frame top hiển thị")
    return p.parse_args()


if __name__ == "__main__":
    args = parse_args()
    summarize(args.input, args.top)

Overwriting /content/drive/MyDrive/LAB5/analytics_spark.py


In [16]:
import sys
sys.path.insert(0, "/content/drive/MyDrive/LAB5")

from analytics_spark import summarize
summarize("/content/drive/MyDrive/LAB5/output", top_n=5)

✓ Đọc được 56841 bản ghi từ /content/drive/MyDrive/LAB5/output

Thống kê theo camera:
+---------+------------+----------------+-------------+----------+
|camera_id|total_frames|total_detections|avg_per_frame|peak_count|
+---------+------------+----------------+-------------+----------+
|NULL     |0           |NULL            |NULL         |NULL      |
|A1       |420         |4001            |9.53         |22        |
+---------+------------+----------------+-------------+----------+


Khung giờ đông người nhất:
+---------+----+----------+------------+
|camera_id|hour|avg_people|total_people|
+---------+----+----------+------------+
|NULL     |NULL|NULL      |NULL        |
|A1       |0   |9.53      |4001        |
+---------+----+----------+------------+


Top 5 frame đông người nhất:
+---------+--------+------------+--------------------------------+
|camera_id|frame_id|people_count|processed_ts                    |
+---------+--------+------------+--------------------------------+
|A1  